# Colab 02 — One baseline experiment

## Goal

Stage one dataset, run one explicitly configured GCN/GAT/GIN baseline experiment, and copy its checkpoint plus lightweight records back to Drive.

**Training and synchronisation default to off.**


## Setup


In [ ]:
from pathlib import Path
from importlib import import_module
import json
import os
import shutil
import subprocess
import sys

GIT_URL = "https://github.com/Thebluesky303/msc-dissertation-drug-repurposing.git"
BRANCH = "v2-refactor"
REPO_DIR = Path("/content/msc-dissertation-drug-repurposing")
WORKSPACE_DIR = Path("/content/dti_workspace")
DRIVE_ROOT = Path("/content/drive/MyDrive/msc-dissertation-drug-repurposing")


def configure_workspace_environment() -> None:
    # Configure all paths before importing config or any src module.
    os.environ["AML_GNN_PROJECT_ROOT"] = str(REPO_DIR)
    os.environ["AML_GNN_DATA_DIR"] = str(WORKSPACE_DIR / "data")
    os.environ["AML_GNN_MODELS_DIR"] = str(WORKSPACE_DIR / "checkpoints")
    os.environ["AML_GNN_RESULTS_DIR"] = str(WORKSPACE_DIR / "results")
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))


def require_api(module_name: str, attribute: str):
    module = import_module(module_name)
    if not hasattr(module, attribute):
        raise RuntimeError(f"Missing required v2 API: {module_name}.{attribute}")
    return getattr(module, attribute)


def copy_directory(source: Path, destination: Path) -> None:
    if not source.exists():
        raise FileNotFoundError(source)
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(source, destination, dirs_exist_ok=True)


print(json.dumps({
    "git_url": GIT_URL,
    "branch": BRANCH,
    "repo": str(REPO_DIR),
    "workspace": str(WORKSPACE_DIR),
    "drive_root": str(DRIVE_ROOT),
}, indent=2))


In [ ]:
DATASET = "davis"       # davis | kiba
SPLIT = "random"        # random | cold_drug | cold_target
MODEL = "gcn"           # gcn | gat | gin
SEED = 42               # 42 | 123 | 2026

RUN_STAGE_DATA = False
RUN_TRAINING = False
RUN_SYNC_TO_DRIVE = False


## Steps


In [ ]:
if RUN_STAGE_DATA:
    copy_directory(
        DRIVE_ROOT / "data" / "raw" / DATASET,
        WORKSPACE_DIR / "data" / "raw" / DATASET,
    )
    if (REPO_DIR / "data" / "splits").exists():
        copy_directory(REPO_DIR / "data" / "splits", WORKSPACE_DIR / "data" / "splits")
    if (DRIVE_ROOT / "data" / "splits").exists():
        copy_directory(DRIVE_ROOT / "data" / "splits", WORKSPACE_DIR / "data" / "splits")
else:
    print("Data staging skipped: RUN_STAGE_DATA is False.")


In [ ]:
experiment_record = None
if RUN_TRAINING:
    os.environ["AML_GNN_PROJECT_ROOT"] = str(REPO_DIR)
    os.environ["AML_GNN_DATA_DIR"] = str(WORKSPACE_DIR / "data")
    os.environ["AML_GNN_MODELS_DIR"] = str(WORKSPACE_DIR / "checkpoints")
    os.environ["AML_GNN_RESULTS_DIR"] = str(WORKSPACE_DIR / "results")
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))

    from config import ExperimentConfig, SPLITS_DIR
    run_baseline_experiment = require_api("src.training", "run_baseline_experiment")
    experiment = ExperimentConfig(
        dataset=DATASET, split=SPLIT, model=MODEL, seed=SEED
    )
    manifest_path = SPLITS_DIR / "manifests" / (
        f"{experiment.dataset}_{experiment.split}_seed{experiment.seed}_manifest.csv"
    )
    experiment_record = run_baseline_experiment(
        config=experiment,
        split_manifest_path=manifest_path,
    )
    print(json.dumps(experiment_record, indent=2, default=str))
else:
    print("Baseline training skipped: RUN_TRAINING is False.")


In [ ]:
if RUN_SYNC_TO_DRIVE:
    if experiment_record is None:
        raise RuntimeError("Run and inspect the experiment before syncing.")
    for name in ("checkpoints", "results"):
        source = WORKSPACE_DIR / name
        if source.exists():
            copy_directory(source, DRIVE_ROOT / name)
    if (WORKSPACE_DIR / "data" / "splits").exists():
        copy_directory(
            WORKSPACE_DIR / "data" / "splits",
            DRIVE_ROOT / "data" / "splits",
        )
    print("Durable outputs synced to:", DRIVE_ROOT)
else:
    print("Drive sync skipped: RUN_SYNC_TO_DRIVE is False.")


## Checks

Verify validation-MSE checkpoint selection, dataset-specific score labels, final test isolation, and the full structured experiment record before synchronising.

## Next steps

Change only one dataset/split/model/seed configuration at a time.
